In [10]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import (mean_absolute_error, mean_squared_error,
                             r2_score, mean_absolute_percentage_error)
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer

Datatype = "/Users/padakandlarohith/Downloads/World Energy Consumption.csv"
Target = "primary_energy_consumption"
SPLITYEAR = 2015
RANDOMSTATE = 42

FEATURES = [
    "year", "population", "gdp",
    "coal_production", "oil_production", "gas_production",
    "electricity_generation", "electricity_demand",
    "greenhouse_gas_emissions",
]
# NOTE: log1p on features requires non-negative values; clip in case of odd data


# ---------------------------------------------------------------- data
df = pd.read_csv(Datatype)
df = df[df["iso_code"].notna() & (df["year"] >= 1985)]      # countries only
df = df.dropna(subset=[Target])
df = df[df[Target] > 0]
print(f"Rows used: {len(df):,} | countries: {df['country'].nunique()} | "
      f"years: {df['year'].min()}-{df['year'].max()}")

df = df.copy()
train = df[df["year"] <= SPLITYEAR]
test = df[df["year"] > SPLITYEAR]
X_train, y_train = train[FEATURES].clip(lower=0), train[Target]
X_test, y_test = test[FEATURES].clip(lower=0), test[Target]
print(f"Train: {len(train):,} rows | Test: {len(test):,} rows\n")


# ---------------------------------------------------------------- models
def make_pipeline(estimator, scale=False):
    steps = []
    if scale:  # linear models need log-scaled features (inputs are heavily right-skewed)
        steps.append(("log", FunctionTransformer(np.log1p, validate=False)))
    steps.append(("imputer", SimpleImputer(strategy="median", add_indicator=True)))
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", estimator))
    # train on log1p(target), predict back on original scale
    return TransformedTargetRegressor(Pipeline(steps), func=np.log1p, inverse_func=np.expm1)


models = {
    "Baseline (median)": make_pipeline(DummyRegressor(strategy="median")),
    "Ridge": make_pipeline(Ridge(alpha=1.0), scale=True),
    "Random Forest": make_pipeline(
        RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=RANDOMSTATE)),
    "HistGradientBoosting": make_pipeline(
        HistGradientBoostingRegressor(random_state=RANDOMSTATE)),
}


# ---------------------------------------------------------------- metrics
def evaluate(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return {
        "MAE (TWh)": mean_absolute_error(y_true, y_pred),
        "RMSE (TWh)": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE (%)": 100 * mean_absolute_percentage_error(y_true, y_pred),
        "RMSLE": np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred))),
    }


# ---------------------------------------------------------------- hold-out test
results, preds = {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds[name] = model.predict(X_test)
    results[name] = evaluate(y_test, preds[name])

results_df = pd.DataFrame(results).T.sort_values("RMSLE")
print(f"=== Hold-out test (years > {SPLITYEAR}) ===")
print(results_df.round(3).to_string(), "\n")

# ---------------------------------------------------------------- group CV
# Every fold holds out entire countries -> tests generalisation to unseen countries.
cv = GroupKFold(n_splits=5)
cv_rows = {}
for name, model in models.items():
    scores = cross_validate(
        model, df[FEATURES].clip(lower=0), df[Target], groups=df["country"], cv=cv,
        scoring={"r2": "r2", "neg_mae": "neg_mean_absolute_error"}, n_jobs=-1)
    cv_rows[name] = {
        "CV R2 mean": scores["test_r2"].mean(),
        "CV R2 std": scores["test_r2"].std(),
        "CV MAE mean": -scores["test_neg_mae"].mean(),
    }
print("=== 5-fold GroupKFold CV (by country) ===")
print(pd.DataFrame(cv_rows).T.round(3).to_string(), "\n")

# ---------------------------------------------------------------- best model diagnostics
best = results_df.index[0]
if best.startswith("Baseline"):
    best = results_df.index[1]
y_pred = np.clip(preds[best], 0, None)
resid = y_test.values - y_pred
print(f"Best model: {best}")

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

lim = [min(y_test.min(), y_pred.min()) + 1e-1, max(y_test.max(), y_pred.max())]
axes[0].scatter(y_test, y_pred, s=8, alpha=0.5)
axes[0].plot(lim, lim, "r--", lw=1)
axes[0].set(xscale="log", yscale="log", xlabel="Actual (TWh)", ylabel="Predicted (TWh)",
            title=f"{best}: actual vs predicted")

axes[1].scatter(y_pred, resid, s=8, alpha=0.5)
axes[1].axhline(0, color="r", ls="--", lw=1)
axes[1].set(xscale="log", xlabel="Predicted (TWh)", ylabel="Residual (TWh)",
            title="Residuals")

axes[2].hist(np.log1p(y_test) - np.log1p(y_pred), bins=50)
axes[2].set(xlabel="log-error", title="Log-scale error distribution")

plt.tight_layout()
plt.savefig("/Users/padakandlarohith/Downloads/model_evaluation_plots.png", dpi=130)

# worst countries
err = pd.DataFrame({"country": test["country"].values, "abs_err": np.abs(resid)})
worst = (err.groupby("country")["abs_err"].mean().sort_values(ascending=False).head(10))
print("\nTop 10 countries by mean absolute error (TWh):")
print(worst.round(1).to_string())

# ---------------------------------------------------------------- permutation importance
from sklearn.inspection import permutation_importance
imp = permutation_importance(models[best], X_test, y_test, n_repeats=5,
                             random_state=RANDOMSTATE, scoring="r2", n_jobs=-1)
imp_df = pd.Series(imp.importances_mean, index=FEATURES).sort_values(ascending=False)
print("\nPermutation importance (drop in R2 on test set):")
print(imp_df.round(4).to_string())

results_df.to_csv("/Users/padakandlarohith/Downloads/model_evaluation_results.csv")


Rows used: 7,837 | countries: 216 | years: 1985-2022
Train: 6,484 rows | Test: 1,353 rows

=== Hold-out test (years > 2015) ===
                      MAE (TWh)  RMSE (TWh)     R2  MAPE (%)  RMSLE
HistGradientBoosting    415.016    2588.646  0.475    45.981  0.957
Random Forest           159.002    1215.840  0.884    34.074  1.162
Ridge                   652.501    3256.923  0.169   104.693  1.399
Baseline (median)       812.760    3658.450 -0.049  2043.904  2.419 

=== 5-fold GroupKFold CV (by country) ===
                      CV R2 mean  CV R2 std  CV MAE mean
Baseline (median)         -0.147      0.054      589.754
Ridge                      0.301      0.788      398.469
Random Forest              0.865      0.196      215.301
HistGradientBoosting       0.841      0.182      227.201 

Best model: HistGradientBoosting

Top 10 countries by mean absolute error (TWh):
country
China            26336.4
United States    13746.3
India             4724.5
Russia            3984.2
Japan       